In [ ]:
#IMPORT LIBRARY
import numpy as np 
import pandas as pd
import segyio

import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

import torch
import torch.nn.functional as F 
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader 
from torchsummary import summary

import pandas as pd
from scipy.interpolate import griddata 
from tqdm.auto import tqdm
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error 
from sklearn.metrics import mean_absolute_error

In [ ]:
import numpy as np
import segyio
import os

# Path file SEGY
#segy_file = r"C:\Users\azkaa\Downloads\PCI_3D_Orig.sgy"
segy_file = r"D:\KP Petrel ya\Data seismik\PCI_3D_Orig.sgy"

# Buka file SEGY
with segyio.open(segy_file, "r", ignore_geometry=True) as f:
    f.mmap()

    # Ambil waktu asli dalam milidetik
    time = f.samples.astype(np.float32)  # time.shape = (n_samples,)
    
    # Total jumlah trace
    n_traces = f.tracecount

    # Ambil header untuk inline dan crossline
    inlines = np.array([f.header[i][5] for i in range(n_traces)], dtype=int)
    crosslines = np.array([f.header[i][21] for i in range(n_traces)], dtype=int)

    # Unique nilai inline dan crossline
    iline_vals = np.unique(inlines)
    xline_vals = np.unique(crosslines)

    # Buat bentuk array data: (n_inline, n_xline, n_time)
    shape = (len(iline_vals), len(xline_vals), len(time))
    print("Shape data cube:", shape)

    # Buat array kosong
    data = np.zeros(shape, dtype=np.float32)

    # Mapping dari nilai inline/xline ke indeks
    iline_idx = {val: i for i, val in enumerate(iline_vals)}
    xline_idx = {val: i for i, val in enumerate(xline_vals)}

    # Isi array dengan data trace
    for i in range(n_traces):
        il = inlines[i]
        xl = crosslines[i]
        if il in iline_idx and xl in xline_idx:
            data[iline_idx[il], xline_idx[xl], :] = f.trace[i]

# Simpan ke file .npz
#output_path = r"C:\Users\azkaa\Downloads\seismic_data.npz"
output_path = r"D:\KP Petrel ya\Data seismik\seismic_data.npz"
np.savez_compressed(output_path,
                    data=data,
                    inline=iline_vals,
                    crossline=xline_vals,
                    time=time)

print(f"✅ Konversi selesai. Data disimpan di '{os.path.basename(output_path)}'")


In [ ]:
print(time[:10])  # misalnya output: [0. 2. 4. 6. 8. 10. 12. 14. 16. 18.]
print(time[-5:])  # misalnya output: [1042. 1044. 1046. 1048. 1050.]
print(data)
print(data.shape)


In [ ]:
import numpy as np
from tqdm import tqdm


# Import file .npz
#dataSG = np.load(r"C:\Users\azkaa\Downloads\seismic_data.npz")['data']
dataSG = np.load(r"D:\KP Petrel ya\Data seismik\seismic_data.npz")['data']
# Scaled Data
scaled_data = 2 * (dataSG - np.min(dataSG)) / (np.max(dataSG) - np.min(dataSG)) - 1

min_inline = 1300
max_inline = 1631
min_xline = 5600 
max_xline = 5851
min_time = 0
max_time = 2100

#file_path = r"C:\Users\azkaa\Downloads\U1_Charisma3D.txt"
file_path = r"D:\KP Petrel ya\Data seismik\U1_Charisma3D.txt"
#lines = np.loadtxt(r"C:\Users\azkaa\Downloads\converted_picking.txt",skiprows=1)
lines = np.loadtxt(r"D:\KP Petrel ya\Data seismik\converted_picking.txt",skiprows=1)

from sklearn.impute import SimpleImputer 
import numpy as np 
import matplotlib.pyplot as plt 
# Load the horizon data from the txt file 
# Extract the coordinates and depth values from the data 
x = lines[:,1]  # inline values 
y = lines[:,0]  # xline values 
depth_values = lines[:, -1] 


In [ ]:
print(dataSG.shape)

In [ ]:
# Create a 2D array to hold the horizon data with missing values set to 0 
t= np.full((332, 252), np.nan) 
for i in tqdm (range(len(lines))): 
    x_indices = int(x[i]-min_xline) 
    # print(x_indices) 
    y_indices = int(y[i]-min_inline) 
    # print(y_indices) 
    t[y_indices, x_indices] = int(((depth_values[i]) / max_time) * data.shape[2]) 
# imputer = SimpleImputer(strategy='mean') 
# t = imputer.fit_transform(t) 
# Plot the horizon data as a surface with missing values set to 0 
print(t)
plt.imshow(t,interpolation='nearest', aspect='auto', origin="lower") 
plt.colorbar(fraction=0.0318, pad=0.04) 
plt.xlabel('xlines') 
plt.ylabel('inlines') 
plt.show()

#print(t[244,:])

In [ ]:
#hor = pd.read_csv(r"C:\Users\azkaa\Downloads\hor_new.csv",skiprows=1)
hor = pd.read_csv(r"D:\KP Petrel ya\Data seismik\hor_new.csv",skiprows=1)

xi = np.linspace(min_xline,max_xline,251) 
yi = np.linspace(min_inline,max_inline,332) 

# Create meshgrid for interpolation 
x_grid, y_grid = np.meshgrid(xi, yi) 
xline = hor.iloc[:, 1] 
#print(xline)
inline = hor.iloc[:, 0] 
#print(inline)
z_grid = griddata((xline,inline), hor.iloc[:, -1], (x_grid, y_grid) ) 

# Horizon Plot
plt.figure(figsize=(10, 6))
plt.imshow(z_grid, origin="lower")
plt.xlabel('xlines')
plt.ylabel('inlines')
plt.title('Interpolated Horizon')
plt.colorbar()
plt.show()

# Seismic Inline Plot with Horizon
plt.figure(figsize=(8,12))
plt.imshow(np.transpose(data[244,:,:]), aspect = 'auto', cmap='seismic') #extent=[0, 2000, 2100, 0])
print(data)

#xline = np.linspace(0, 2000, t.shape[1])  # atau nilai xline asli jika ada
plt.plot(t[244, :], 'green', linewidth=3.5)
print(t[250,:])
#print(t.shape)
#print(t[244,:].shape)

plt.title('Inline 244', fontsize=22, y=1.04)
plt.ylabel('Time (ms)', fontsize=20)
plt.xlabel('Xline', fontsize=20)
plt.colorbar(fraction=0.023, pad=0.04)
plt.show()


In [ ]:
(inlines, xlines, times) = data.shape
seisdata = data
horlabel = t/2100
seisdata.shape, horlabel.shape
seisdata = seisdata.transpose(0,2,1)
seisdata.shape

def chain_ranges(*args):
    result = []
    for start, stop, step in args:
        result.extend(range(start, stop, step))
    return result
    
val_inline = [17, 89, 175, 35, 87]
train = chain_ranges((1, 150, 10), (160, 320, 10))
# Mengambil indeks yang tidak dihindari
train_inline = np.setdiff1d(train, val_inline)
test_inline = [119, 44, 219, 194]

In [ ]:
#PATCHING
class seismic_dataset(Dataset):
    def __init__(self, indexdata, data, label, width): 
        super().__init__()
        self.idx = indexdata 
        self.data = data 
        self.label = label 
        self.width = width

    def __len__(self):
        total_patches = len(self.idx) * (self.data.shape[-1] - self.width + 1)
        return total_patches

    def __getitem__(self, index):
        section_idx = index // (self.data.shape[-1] - self.width + 1)
        inline_idx = self.idx[section_idx]
        start_col = index % (self.data.shape[-1] - self.width + 1) 
        end_col = start_col + self.width

        data_slice = torch.from_numpy(self.data[inline_idx, :, start_col:end_col])
        label_slice = torch.from_numpy(self.label[inline_idx, start_col:end_col])
        data_slice = data_slice.squeeze().unsqueeze(0)

        return data_slice, label_slice


In [ ]:
index = 6000
width = 10
data_shape_last_dim = seisdata.shape[-1]

section_idx = index // (data_shape_last_dim - width + 1)
inline_idx = train_inline[section_idx]

print("Index:", index)
print("Section index:", section_idx)
print("Inline index:", inline_idx)


In [ ]:
N = len(train_inline)
n_xline = seisdata.shape[0]
n_per_inline = n_xline - 10 + 1  # misalnya width = 10
total_samples = N * n_per_inline

print(f"Total sample kemungkinan: {total_samples}")
print(n_per_inline)
print("Min train_inline:", train_inline.min())
print("Max train_inline:", train_inline.max())
print("seisdata inline dim:", seisdata.shape[0])
print("horlabel inline dim:", horlabel.shape[0])

patch_per_inline = 251 - 10 + 1  # 242

inline_pos = np.where(train_inline == 244)[0]

if len(inline_pos) == 0:
    print("Inline 244 tidak termasuk dalam train_inline")
else:
    inline_pos = inline_pos[0]
    sample_start = inline_pos * patch_per_inline
    sample_end = sample_start + patch_per_inline - 1
    print(f"Inline 244 ada di posisi ke-{inline_pos} dalam train_inline")
    print(f"Sample index mulai dari {sample_start} sampai {sample_end}")


In [ ]:
train_data = seismic_dataset(train_inline, seisdata, horlabel, width = 10)
sampledata, samplelbl = train_data[5000] 
#print(sampledata.shape, samplelbl.shape) 
print(samplelbl.shape)
print(samplelbl.squeeze().shape)
print(samplelbl)

plt.figure(figsize=(10,8)) 
plt.imshow(sampledata.squeeze().numpy(),interpolation='none',cmap= 'seismic', aspect='auto', vmin=-1, vmax=1) 
# 


#plt.imshow(sampledata.squeeze().numpy(),interpolation='none',cmap= 'seismic',extent=[0,200,500,0], vmin=-1, vmax=1) 
plt.plot(samplelbl.squeeze().numpy() *2100, 'yellow', linewidth=3, linestyle='-') 
plt.colorbar(fraction=0.03, pad=0.04) 
plt.show() 

plt.figure(figsize=(10,8)) 
plt.imshow(seisdata[244,:,:], interpolation='none', cmap='seismic', aspect='equal')
plt.plot(horlabel[244,:] * 2100, 'cyan', linewidth=2.5, linestyle='-')
plt.colorbar(fraction=0.023, pad=0.04) 
plt.show()

In [ ]:
print(horlabel.shape)
print(horlabel[1])
print(seisdata.shape)
print(seisdata[0])

In [ ]:
inline_idx = 244  # dari perhitungan patch 6000
start_col = 192
end_col = start_col + 10

print(horlabel[inline_idx, start_col:end_col])


In [ ]:
#TRAINING

import timm 
class seismic_model(nn.Module): 
    def __init__(self, num_channel, num_output): 
        super(seismic_model,self).__init__() 
        self.model=timm.create_model('resnet34', pretrained=False) 
        self.model.conv1=nn.Conv2d(num_channel, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False) 
        self.model.fc=nn.Sequential( 

nn.Linear(in_features=self.model.num_features,out_features=256), 
nn.ReLU(), 
nn.Dropout(0.1),  # Adding dropout for regularization 
nn.Linear(in_features=256, out_features=num_output)  #Output layer for regression 
    ) 
    def forward(self,x): 
        out=self.model(x) 
        return out

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')     
model = seismic_model(num_channel=1, num_output=10) 
model.to(device) 
summary(model, input_size=(1, 526, 10)) 


In [ ]:
import os
import csv
from tqdm import tqdm
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import mean_squared_error

batch_size = 4
batch_size_test = batch_size
num_epochs = 8
learning_rate = 1e-4

# Create dataset
train_data = seismic_dataset(train_inline, seisdata, horlabel, width=10)
test_data = seismic_dataset(test_inline, seisdata, horlabel, width=10)

print(len(train_data), len(test_data))

# Filter out NaNs
filtered_train_data = [(data, label) for data, label in train_data if not torch.isnan(label).any()]
filtered_test_data = [(data, label) for data, label in test_data if not torch.isnan(label).any()]

print(len(filtered_train_data), len(filtered_test_data))

train_loader = DataLoader(filtered_train_data, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(filtered_test_data, batch_size=batch_size, shuffle=False, drop_last=True)

criterion = torch.nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

train_losses = []
valid_losses = []
train_mse = []
test_mse = []
mse_old = 0.0

#csv_path = r"C:\Users\azkaa\OneDrive\Dokumen\kuliah\Geoseismal\Project\training_coba.csv"
csv_path = r"D:\KP Petrel ya\Data seismik\training_coba.csv"

# Paths for saving models
#best_model_path = r"C:\Users\azkaa\OneDrive\Dokumen\kuliah\Geoseismal\Project\Model_Best\model_best.ckpt"
best_model_path = r"D:\KP Petrel ya\Data seismik\Model_best\model_best.ckpt"
#last_model_path = r"C:\Users\azkaa\OneDrive\Dokumen\kuliah\Geoseismal\Project\Model_Last\model_last.ckpt"
last_model_path = r"D:\KP Petrel ya\Data seismik\Model_last\model_last.ckpt"

# Make sure directories exist
os.makedirs(os.path.dirname(best_model_path), exist_ok=True)
os.makedirs(os.path.dirname(last_model_path), exist_ok=True)

with open(csv_path, 'w', newline='') as csvfile:
    fieldnames = ['Epoch', 'Train_loss', 'Valid_loss', 'Train_mse', 'Test_mse']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    
    for epoch in range(1, num_epochs + 1):
        train_loss = 0.0
        valid_loss = 0.0
        mse_train = 0.0
        mse_test = 0.0

        model.train()
        for data, label in tqdm(train_loader):
            data = data.to(device)
            label = label.to(device)
            optimizer.zero_grad()
            output = model(data.float())
            loss = criterion(output, label)
            mse = mean_squared_error(
                label.squeeze().detach().cpu().numpy().flatten(),
                output.squeeze().detach().cpu().numpy().flatten()
            )
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * data.size(0)
            mse_train += mse

        model.eval()
        for data, label in tqdm(test_loader):
            data = data.to(device)
            label = label.to(device)
            output = model(data.float())
            loss = criterion(output, label)
            mse = mean_squared_error(
                label.squeeze().detach().cpu().numpy().flatten(),
                output.squeeze().detach().cpu().numpy().flatten()
            )
            valid_loss += loss.item() * data.size(0)
            mse_test += mse

        train_loss = train_loss / len(train_loader.sampler) * batch_size
        mse_train = mse_train / len(train_loader.sampler) * batch_size
        valid_loss = valid_loss / len(test_loader.sampler) * batch_size_test
        mse_test = mse_test / len(test_loader.sampler) * batch_size_test

        train_losses.append(train_loss)
        valid_losses.append(valid_loss)
        train_mse.append(mse_train)
        test_mse.append(mse_test)

        if mse_old < mse_test:
            mse_old = mse_test
            torch.save(model.state_dict(), best_model_path)

        writer.writerow({
            'Epoch': epoch,
            'Train_loss': train_loss,
            'Valid_loss': valid_loss,
            'Train_mse': mse_train,
            'Test_mse': mse_test,
        })

        print('Epoch: {} \tTrain Loss: {:.7f} \tVal. Loss: {:.7f} \tmse_train: {:.7f} \tmse_test: {:.7f}'.format(
            epoch, train_loss, valid_loss, mse_train, mse_test))

        # Save last model every epoch
        torch.save(model.state_dict(), last_model_path)


In [ ]:
# Read CSV file into a pandas DataFrame
#df = pd.read_csv(r"C:\Users\azkaa\OneDrive\Dokumen\kuliah\Geoseismal\Project\training_coba.csv")
df = pd.read_csv(r"D:\KP Petrel ya\Data seismik\training_coba.csv")
print(df.head())  # Lihat isi awal dataframe
print(df[['Train_loss', 'Valid_loss', 'Train_mse', 'Test_mse']].describe())  # Cek ringkasan statistik
epoch = df['Epoch']
train_losses = df['Train_loss']
valid_losses = df['Valid_loss']
train_mse = df['Train_mse']
test_mse = df['Test_mse']

# Buat subplots 
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 3))
plt.subplots_adjust(wspace=0.3) # Atur jarak antar subplot

# Plot untuk loss
ax1.plot(train_losses, label='Training loss')
ax1.plot(valid_losses, label='Validation loss')
ax1.set_xlabel('Epoch', fontsize=16)
ax1.set_ylabel('Loss', fontsize=16)
ax1.set_title('Loss', fontsize=18, y=1.05)
ax1.legend(['Train', 'Test'], fontsize=12)
ax1.set_ylim(0, 0.095)

# Plot untuk MSE
ax2.plot(train_mse, label='Train MSE')
ax2.plot(test_mse, label='Test MSE')
ax2.set_xlabel('Epoch', fontsize=16)
ax2.set_ylabel('MSE', fontsize=16)
ax2.set_title('MSE', fontsize=18, y=1.05)
ax2.legend(['Train', 'Test'], fontsize=12)
ax2.set_ylim(0, 0.0015)

# Sesuaikan layout
plt.tight_layout(pad=1)

# plot
plt.show()

# Load model
#model.load_state_dict(torch.load(r"C:\Users\azkaa\OneDrive\Dokumen\kuliah\Geoseismal\Project\Model_Last\model_last.ckpt"))
model.load_state_dict(torch.load(r"D:\KP Petrel ya\Data seismik\Model_last\model_last.ckpt"))

#Garis Train itu ngeplot "Train Loss", bukan "validation loss"

In [ ]:
print(horlabel.min(), horlabel.max())

In [ ]:
# PROSES PREDIKSI

patch_size = 10
stride = 8
output_array = []
model.eval()
print()

datapred = seisdata.transpose(0, 1, 2)  # (inline, time, xline)
print("datapred shape:", datapred.shape)

# Initialize the prediction array
# Menggunakan ukuran sesuai dimensi asli horlabel (inline, xline)
t_predd = np.zeros((datapred.shape[0], horlabel.shape[1]))

# Loop over all inlines
for inline in range(datapred.shape[0]):
    val_data = datapred[inline, :, :]  # shape (time, xline)
    val_label = horlabel[inline, :]   # shape (xline,)

    id_result = []
    result = []
    loop = 0

    # main patch
    for i in range(0, val_data.shape[1] - patch_size + 1, stride):
        patch_data = val_data[:, i:i+patch_size]
        patch_label = val_label[i:i+patch_size]

        data_tensor = torch.from_numpy(patch_data).unsqueeze(0).unsqueeze(0).float()
        label_tensor = torch.from_numpy(patch_label).unsqueeze(0).float()

        with torch.no_grad():
            output = model(data_tensor)
            out_reg = output.squeeze().detach().cpu().numpy().flatten()

        id_result.extend(np.arange(i, i+patch_size).tolist())
        result.extend(out_reg.tolist())
        loop += 1

    # last pach
    last_start = val_data.shape[1] - patch_size
    if last_start % stride != 0:
        patch_data = val_data[:, last_start:last_start+patch_size]
        patch_label = val_label[last_start:last_start+patch_size]

        data_tensor = torch.from_numpy(patch_data).unsqueeze(0).unsqueeze(0).float()
        label_tensor = torch.from_numpy(patch_label).unsqueeze(0).float()

        with torch.no_grad():
            output = model(data_tensor)
            out_reg = output.squeeze().detach().cpu().numpy().flatten()

        id_result.extend(np.arange(last_start, last_start+patch_size).tolist())
        result.extend(out_reg.tolist())
        loop += 1

    pd_result = pd.DataFrame({'id': id_result, 'hor': result})  # Buat DataFrame untuk agregasi rata-rata prediksi patch overlapping
    prediction = pd_result.groupby('id').mean()['hor'].values  # Kelompokkan per id dan rata-rata hasil prediks

    # PENTING: assign prediction ke t_predd dengan panjang yang cocok
    length_pred = len(prediction)
    length_label = t_predd.shape[1]
    
    if length_pred < length_label:
        # Jika prediksi lebih pendek, isi sebagian, sisanya NaN atau 0
        t_predd[inline, :length_pred] = prediction
        t_predd[inline, length_pred:] = np.nan  # Bisa juga 0 sesuai kebutuhan
    else:
        # Jika sama atau lebih panjang (biasanya sama), assign langsung
        t_predd[inline, :] = prediction[:length_label]

# Simpan hasil prediksi
#np.save(r"C:\Users\azkaa\OneDrive\Dokumen\kuliah\Geoseismal\Project\Model_Predict.npy", t_predd)
np.save(r"D:\KP Petrel ya\Data seismik\Model_Predict.npy", t_predd)

# Post-processing jika perlu
t_pred = t_predd

# Contoh normalisasi hasil (bisa disesuaikan)
t_pred_flattened = t_pred.flatten()
t_pred_flattened[t_pred_flattened == 0.02262795] = np.nan
t_pred = t_pred_flattened.reshape(t_pred.shape)

# Optional: handling indeks zero pada data
zero_indices = np.where(np.all(seisdata == 0.02262795, axis=1))
t_pred[zero_indices] = np.nan
z_grid[zero_indices] = np.nan

print("Prediction selesai!")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.mplot3d import Axes3D

#path_predict = r"C:\Users\azkaa\OneDrive\Dokumen\kuliah\Geoseismal\Project\Model_Predict.npy"
path_predict = r"D:\KP Petrel ya\Data seismik\Model_Predict.npy"
t_predd = np.load(path_predict)
# Samain ukuran
min_rows = min(t.shape[0], t_pred.shape[0])
min_cols = min(t.shape[1], t_pred.shape[1])

t_trimmed = t[:min_rows, :min_cols] * 4
t_pred_trimmed = t_pred[:min_rows, :min_cols] *2100 *4

# Masking di area label
valid_mask = ~np.isnan(t_trimmed) & (t_trimmed != 0)
t_pred_masked = np.where(valid_mask, t_pred_trimmed, np.nan)

# ========== PENAMPANG 2D ==========
example = 126  # baris inline dicek
plt.figure(figsize=(7, 7))
plt.imshow(seisdata[example, :], cmap='seismic', extent=[0,250,2100,0], aspect="auto")
plt.plot(t_pred_masked[example, :], c='yellow', label='Prediksi')
plt.plot(t_trimmed[example, :], c='cyan', label='Ground Truth')
plt.xlabel('Xline', fontsize=19)
plt.ylabel('Time (ms)', fontsize=19)
plt.title(f'Inline {example}', fontsize=20, y=1.04)
plt.legend()
plt.show()

# ========== TAMPAK ATAS ==========
fig, axs = plt.subplots(1, 2, figsize=(12, 12))

im0 = axs[0].imshow(t_trimmed, interpolation='nearest', cmap='viridis', origin='lower', aspect='auto')
axs[0].set_title('Ground Truth (Picking Manual)', fontsize=22, y=1.04)
axs[0].set_xlabel('Xline', fontsize=21)
axs[0].set_ylabel('Inline', fontsize=21)

im1 = axs[1].imshow(t_pred_masked, interpolation='nearest', cmap='viridis', origin='lower', aspect='auto')
axs[1].set_title('Prediksi (Masked)', fontsize=22, y=1.04)
axs[1].set_xlabel('Xline', fontsize=21)
axs[1].set_ylabel('Inline', fontsize=21)

colorbar = plt.colorbar(im0, ax=axs, fraction=0.020, pad=0.02)
colorbar.set_label('Time (ms)', fontsize=20)
plt.show()

# ========== VISUALISASI 3D ==========
fig = plt.figure(figsize=(30, 20))

X, Y = np.meshgrid(range(min_cols), range(min_rows))

# Plot Ground Truth
ax1 = fig.add_subplot(131, projection='3d')
ax1.plot_surface(X, Y, t_trimmed, cmap='viridis_r')
ax1.set_title('Ground Truth (Picking Manual)', fontsize=24, y=0.94)
ax1.set_xlabel('Xline', labelpad=9, fontsize=21)
ax1.set_ylabel('Inline', labelpad=9, fontsize=21)
ax1.set_zlabel('Time (ms)', rotation=90, labelpad=6, fontsize=18)
ax1.invert_zaxis()
ax1.view_init(elev=20, azim=-140)
ax1.set_zlim(1540, 1360)
ax1.dist = 11.6

# Plot Prediksi (Masked)
ax2 = fig.add_subplot(132, projection='3d')
ax2.plot_surface(X, Y, t_pred_masked, cmap='viridis_r')
ax2.set_title('Prediksi', fontsize=24, y=0.94)
ax2.set_xlabel('Xline', labelpad=9, fontsize=21)
ax2.set_ylabel('Inline', labelpad=9, fontsize=21)
ax2.set_zlabel('Time (ms)', rotation=90, labelpad=6, fontsize=18)
ax2.invert_zaxis()
ax2.view_init(elev=20, azim=-140)
ax2.set_zlim(1540, 1360)
ax2.dist = 11.6

plt.subplots_adjust(left=0.2, right=0.9, bottom=0.1, top=0.9, wspace=0.02, hspace=0.12)
plt.show()